# GPT2 Classification post trainning test


In [10]:
import sys
import os

sys.path.append(os.path.abspath("../.."))

import pandas as pd
import tiktoken
import torch
from torch.utils.data import DataLoader
import torch_directml
from Train.load_pretrained_gpt2 import pretrained_gpt2_generator
from Models.config import GPT_CONFIG_124M
from Train.classification_post_train import SentimentDataset

In [21]:
test_df = pd.read_parquet(
    "hf://datasets/stanfordnlp/sst2/data/validation-00000-of-00001.parquet" 
)

tokenizer = tiktoken.get_encoding("gpt2")

test_dataset = SentimentDataset(dataframe=test_df, tokenizer=tokenizer)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0,
    drop_last=True,
)

In [17]:
model = pretrained_gpt2_generator(GPT_CONFIG_124M)
model.out_head = torch.nn.Linear(
    in_features=GPT_CONFIG_124M["emb_dim"], out_features=2
)

epoch_to_load = 5

PATH = "../../weights/GPTModel_epoch_5.pt"

checkpoint = torch.load(PATH, weights_only=False)

model.load_state_dict(checkpoint)

model.eval()

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (transformer_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (Q): Linear(in_features=768, out_features=768, bias=True)
        (K): Linear(in_features=768, out_features=768, bias=True)
        (V): Linear(in_features=768, out_features=768, bias=True)
        (projection): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (feedforward): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (Q): Linear(in_features=768, out_f

In [18]:
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    try:
        device = torch_directml.device()
    except Exception:
        device = torch.device("cpu")
print(f"Using devide: {device}")

model.to(device)

Using devide: privateuseone:0


GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (transformer_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (Q): Linear(in_features=768, out_features=768, bias=True)
        (K): Linear(in_features=768, out_features=768, bias=True)
        (V): Linear(in_features=768, out_features=768, bias=True)
        (projection): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (feedforward): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (Q): Linear(in_features=768, out_f

In [23]:
from Train.classification_post_train import get_last_real_token_logits

correct = 0
examples_seen = 0

with torch.no_grad():
    for i , (input_batch,target_batch) in enumerate(test_loader):
        input_batch = input_batch.to(device)
        target_batch = target_batch.to(device)

        logits = get_last_real_token_logits(input_batch, model, 50256)
        loss = torch.nn.functional.cross_entropy(logits, target_batch)

        preds = torch.argmax(logits, dim=-1)
        correct += (preds == target_batch).sum().item()
        examples_seen += target_batch.size(0)

print(
    f"Test results: {correct / examples_seen * 100:.2f}% accuracy. {correct} correct predictions over {examples_seen} examples"
)

Test results: 90.37% accuracy. 788 correct predictions over 872 examples
